BUILDING AGENTIC RAG PIPLINE

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders  import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore


C:\Users\venktesh mahajan\AppData\Local\Temp\ipykernel_2992\1312136597.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders  import PyPDFLoader


In [ ]:
loader = PyPDFLoader("../apps/Data/Patientreport.pdf")
docs=loader.load()

In [13]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)

In [ ]:
embbedings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
from langchain_chroma import Chroma
vector_store=Chroma(
    
   embedding_function=embbedings, 
   persist_directory="./agentic_Rag.db"

)
vector_store.add_documents(splitted_docs)


Building ai agent 


In [31]:
#for building agent we need tool ,LLM model prompt
from langchain.tools import tool
@tool
def retriver_tool(query:str):
    ''' 
     This tool can help you to retrive the relevant data of pdf documents and these pdfs document have details about medical
    '''
    docs= vector_store.similarity_search(query=query , k=4)
    print("tool used" , query)
    context=""
    for doc in docs:
        context+=doc.page_content + "\n\n"
    return context

    

In [32]:
llm  =ChatGoogleGenerativeAI(model="gemini-3.7-flash")
prompt = """   you are a helpful assistant answer questions using retrieved context ALWAYS use retrieval tool for questions requiring external knowledege  """
from langchain.agents import create_agent

agent=create_agent(model=llm , tools=[retriver_tool] , system_prompt=prompt) 

In [ ]:
query="what is the name of patient and name of doctors"
response=agent.invoke({"messages":[{"role":"user" , "content":query}]})
response["messages"][-1].content